# 02 — Event memory: store → retrieve → inspect

Day 1–3 checks use SQLite and keyword retrieval. No model weights or Groq key are needed. Semantic embeddings remain Day 4 work.

In [ ]:
import sys, subprocess, tempfile
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/app/world.py").exists()), None)
if ROOT is None and "google.colab" in sys.modules:
    ROOT = Path("/content/MiniDavis")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/KTH-Sys/MiniDavis.git", str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "backend/requirements.txt")], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from a MiniDavis checkout.")
sys.path.insert(0, str(ROOT))
print("Project imports ready.")


In [ ]:
from backend.app.memory import EventStore
from backend.app.schemas import Event, StepRequest
from backend.app.world import World
path = Path(tempfile.mkdtemp()) / "memory.sqlite"
store = EventStore(path)
event = Event(event_id="sample:1:ethan", run_id="sample", tick=1, agent_id="ethan", action="gather", outcome="accepted", text="Ethan gathered two wood.")
store.save(event)
store.close()
store = EventStore(path)
assert store.get(event.event_id) == event
print("Stored and reloaded:", event.event_id)


In [ ]:
rejected = Event(event_id="sample:2:ethan", run_id="sample", tick=2, agent_id="ethan", action="move", outcome="rejected", text="Ethan could not move outside the map.")
store.save(rejected)
assert store.get(rejected.event_id).outcome == "rejected"
assert store.get(event.event_id).outcome == "accepted"
context = store.retrieve("sample", "ethan", "wood", as_of_tick=2)
assert event.event_id in [e.event_id for e in context.relevant]
assert not store.retrieve("sample", "gayatri", "wood", 2).recent
print("Accepted/rejected outcomes and private memory scope verified.")


In [ ]:
world = World(seed=11, store=store)
for index in range(2):
    state = world.snapshot()
    state = world.step(StepRequest(run_id=state.run_id, expected_revision=state.revision, request_id=f"memory-{index}"))
for resident in state.agents:
    decision = world.inspect(resident.id).decision
    assert decision.memory.recent
    assert all(store.get(e.event_id) == e for e in decision.memory.recent)
    print(resident.name, [e.event_id for e in decision.memory.recent])
store.close()


These facts are supplied to the agent input. The baseline receives them but uses fixed rules. The Groq adapter uses the bounded subset shown in the inspector; retrieval alone is not proof that memory changed a decision.